# 04_02 - Feature Engineering: Building the Analytical State-Space

## 1. Objective and Theoretical Framework
This notebook executes the final transformation of our consolidated data into a **State-Space Matrix**. To satisfy the **Markov Decision Process (MDP)** requirements, the state at time $t$ must contain all necessary information for optimal decision-making.

### 1.1. Deterministic vs. Stochastic Information (Market Logic)
A critical distinction is made regarding the availability of future information in the Iberian Market (OMIE):
* **Deterministic Spot Price (t+1):** The Day-Ahead auction results for tomorrow are published at 13:00 CET today. Therefore, any operational decision made after this time considers the $D+1$ price as a **known constant**, not an estimate.
* **Stochastic Weather (t+1 to t+3):** Meteorological conditions remain uncertain. We simulate this by providing "clouded" forecasts—realized future values with added Gaussian noise that increases with the forecast horizon (5%, 10%, and 15%).

In [6]:
import pandas as pd
import numpy as np
import sys
import os
from pathlib import Path

# Project root configuration
project_root = Path(os.path.abspath('../../'))
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.config.paths import MERGED_INTERIM_FILE, MODELING_DATASET_FILE
from src.config.constants import TARGET_COLUMN

# Load the merged interim dataset
df = pd.read_csv(MERGED_INTERIM_FILE)
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

print(f"✅ Initial dataset loaded: {df.shape[0]} days.")

✅ Initial dataset loaded: 2192 days.


## 2. Domain-Driven Engineering: Distillation & Foresight

We implement a "Master Mix" strategy to transform raw weather data into supply/demand signals and simulate the availability of future information.

**Key Synthetic Predictors:**
1. **Spot Price (t+1):** Known deterministic price for the next 24 hours.
2. **Thermal Stress:** HDD/CDD derived from temperature observations.
3. **Renewable Regimes:** Solar intensity and High Wind flags.
4. **Noisy Forecasts:** Simulated weather predictions for a 72-hour horizon with incremental Gaussian noise.

In [7]:
# 1. Financial Imputation (OMIP)
future_cols = [col for col in df.columns if 'Future' in col]
df[future_cols] = df[future_cols].bfill().ffill()

# 2. DETERMINISTIC MARKET INFO (The "Scraping" Advantage)
# Since the auction for D+1 is public by 13:00 today, we shift the price backward.
# In production, this is the data directly scraped for "tomorrow".
df['Spot_Price_SPEL_t+1_known'] = df['Spot_Price_SPEL'].shift(-1)

# 3. THERMAL & RENEWABLE DISTILLATION
temp_col = 'apparent_temperature_mean' if 'apparent_temperature_mean' in df.columns else 'temperature_2m_mean'
BASE_TEMP = 20.0
df['HDD'] = np.maximum(0, BASE_TEMP - df[temp_col])
df['CDD'] = np.maximum(0, df[temp_col] - BASE_TEMP)

if 'shortwave_radiation_sum' in df.columns:
    df['solar_intensity'] = df['shortwave_radiation_sum'] / (df['shortwave_radiation_sum'].max() + 1e-9)
    df['is_solar_peak'] = (df['shortwave_radiation_sum'] > 20000).astype(int)

if 'wind_speed_10m_max' in df.columns:
    df['is_high_wind'] = (df['wind_speed_10m_max'] > 20).astype(int)

# 4. STOCHASTIC WEATHER FORESIGHT (D+1 to D+3)
# We take the future realized values and add Gaussian noise proportional to the horizon
forecast_targets = ['solar_intensity', 'HDD', 'CDD', 'is_high_wind']
for col in forecast_targets:
    if col in df.columns:
        for day in [1, 2, 3]:
            future_real = df[col].shift(-day)
            # Noise level: 5% * day (D+1: 5%, D+2: 10%, D+3: 15%)
            noise = np.random.normal(0, future_real.std() * (0.05 * day), size=len(future_real))
            df[f'{col}_pred_t+{day}'] = future_real + noise

# 5. CALENDAR & HOLIDAY STANDARDIZATION
holiday_col = 'is_national_holiday' if 'is_national_holiday' in df.columns else 'is_holiday'
if holiday_col in df.columns:
    df['is_holiday'] = df[holiday_col].fillna(0).astype(int)
    if holiday_col != 'is_holiday': df = df.drop(columns=[holiday_col])
else:
    df['is_holiday'] = 0

df['is_weekend'] = df['date'].dt.dayofweek.isin([5, 6]).astype(int)

# 6. CLEANUP: Dropping raw features used for distillation
raw_weather = [
    'temperature_2m_mean', 'temperature_2m_max', 'temperature_2m_min', 
    'apparent_temperature_mean', 'wind_speed_10m_max', 'shortwave_radiation_sum', 
    'precipitation_sum', 'weather_code'
]
df = df.drop(columns=[c for c in raw_weather if c in df.columns])

# DETERMINISTIC MARKET INFO (D+1)
df['Spot_Price_SPEL_t+1_known'] = df['Spot_Price_SPEL'].shift(-1)

# THE UNKNOWN HORIZON (TARGETS) ---
df['Spot_Price_target_t+2'] = df['Spot_Price_SPEL'].shift(-2)
df['Spot_Price_target_t+3'] = df['Spot_Price_SPEL'].shift(-3)

print("✅ Master Mix and Forecast Proxies successfully generated.")

✅ Master Mix and Forecast Proxies successfully generated.


In [8]:
from src.features.build_feature_matrix import build_feature_matrix

# Integrating with the automated .py module for lags and rolling stats
# Note: We do NOT use dropna() here to preserve row density for Selection
df_final = build_feature_matrix(
    df=df,
    use_time_features=True,
    use_lag_features=True,
    use_rolling_features=True,
    use_future_features=True,
    save=False 
)

print(f"✅ Full Feature Matrix built. Total columns: {df_final.shape[1]}")

✅ Full Feature Matrix built. Total columns: 152


In [9]:
from src.features.build_feature_matrix import build_feature_matrix

print("Executing automated feature engineering pipeline...")

# Note: We do NOT use dropna() here as per our technical decision 
# to let the Feature Selection stage handle sparsity.
df_final = build_feature_matrix(
    df=df,
    use_time_features=True,
    use_lag_features=True,
    use_rolling_features=True,
    use_future_features=True,
    save=False 
)

print(f"✅ Full Feature Matrix built. Total features: {df_final.shape[1]}")

Executing automated feature engineering pipeline...
✅ Full Feature Matrix built. Total features: 152


In [10]:
# Final Export to the processed data layer
MODELING_DATASET_FILE.parent.mkdir(parents=True, exist_ok=True)
df_final.to_csv(MODELING_DATASET_FILE, index=False)

print(f"🚀 State-space matrix (with foresight) saved at:\n{MODELING_DATASET_FILE}")

🚀 State-space matrix (with foresight) saved at:
C:\Users\Alejandro\GitHub\Data-Driven-Strategies-for-Financial-Resilience-in-Energy-Procurement\data\processed\modeling_dataset.csv
